# 🧪 Phase 1: Synthetic Clinical Case Generation & BIO Tagging Demo

This notebook demonstrates the synthetic data generation pipeline for **Kent-AI**:
1. **Rubric Selection**: Querying Kent's Repertory Section 1 (**MIND**) via `KentDB`.
2. **Clinical Vignette Generation**: Creating authentic first-person patient descriptions.
3. **Entity Extraction**: 7-dimension entity spans (`LOC`, `SEN`, `MOD_AGG`, `MOD_AMEL`, `CONC`, `TEMP`, `MENT`).
4. **BIO Auto-Tagging**: Aligning character spans to word tokens and BIO labels for ClinicalBERT.
5. **Stratified Splitting**: Partitioning into reproducible 80/10/10 train/val/test splits.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.kent_db import KentDB, get_rubric_path, get_remedies
from src.data.case_generator import CaseGenerator, SyntheticCase
from src.data.bio_tagger import BIOTagger
from src.data.splitter import split_cases, compute_split_statistics

print('All Kent-AI Phase 1 modules successfully imported!')

## 1. Select 5 Representative MIND Rubrics

We query the digitized Kent's Repertory database for 5 clinically rich MIND rubrics with associated remedies.

In [ ]:
db = KentDB()
sample_rubric_ids = [2, 4, 5, 23, 45]  # Feels abandoned, absent-minded, morning, anxiety, etc.

print(f'{"ID":<5} | {"Rubric Path":<45} | {"Top Remedies"}')
print('-' * 80)
for r_id in sample_rubric_ids:
    path = db.get_rubric_path(r_id)
    rems = db.get_remedies(r_id)
    top_rems = ', '.join([r['abbreviation'] for r in rems[:4]]) or 'None listed'
    print(f'{r_id:<5} | {path:<45} | {top_rems}')

## 2. Generate 5 Synthetic Cases

We initialize `CaseGenerator` in mock mode for offline demonstration (or connect to an active Ollama instance when running with a GPU).

In [ ]:
generator = CaseGenerator(mock_mode=True)
cases = [generator.generate_case_for_rubric(r_id) for r_id in sample_rubric_ids]

for i, case in enumerate(cases, 1):
    print(f'=== CASE {i}: {case.rubric_path} ===')
    print(f'Case ID   : {case.case_id}')
    print(f'Narrative : {case.narrative}')
    print(f'Entities  : {len(case.entities)} identified')
    for ent in case.entities:
        print(f'  - [{ent["label"]}] "{ent["text"]}" (char {ent["start"]}:{ent["end"]})')
    print()

## 3. Verify BIO Auto-Tagger Alignment

Inspect how character-level entity offsets are converted into token sequences and parallel BIO tags (`B-<LABEL>`, `I-<LABEL>`, `O`).

In [ ]:
sample_case = cases[0]
print(f'Narrative: {sample_case.narrative}\n')
print(f'{"Token":<20} | {"BIO Tag":<12}')
print('-' * 35)
for tok, tag in zip(sample_case.tokens[:15], sample_case.bio_tags[:15]):
    print(f'{tok:<20} | {tag:<12}')
if len(sample_case.tokens) > 15:
    print(f'... ({len(sample_case.tokens) - 15} additional tokens)')

## 4. Reproducible 80/10/10 Dataset Stratification

Test dataset partitioning logic ensuring balanced rubric representation across train, validation, and test splits.

In [ ]:
# Generate 20 cases (4 variations for each of the 5 rubrics)
batch_cases = generator.generate_batch(sample_rubric_ids, cases_per_rubric=4)
case_dicts = [c.to_dict() for c in batch_cases]

train_set, val_set, test_set = split_cases(case_dicts, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42)
stats = compute_split_statistics(train_set, val_set, test_set)

print(f'Total Generated Cases: {stats["total_cases"]}')
print(f'Train Split          : {stats["train"]["case_count"]} cases ({stats["train"]["ratio"]*100:.1f}%)')
print(f'Val Split            : {stats["val"]["case_count"]} cases ({stats["val"]["ratio"]*100:.1f}%)')
print(f'Test Split           : {stats["test"]["case_count"]} cases ({stats["test"]["ratio"]*100:.1f}%)')
print('\nTrain Entity Counts  :', stats['train']['entity_counts'])